# Coupled solver example


In [2]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *   # Opencascade for geometry modeling
import pandas as pd
import matplotlib.pyplot as plt
import netgen.occ as occ

def generate_synapse2d(maxh, order_g = 1, external = False):
    wp = occ.WorkPlane()
    wp.Rotate(90).Line(0.1).Rotate(-90)
    wp.Line(0.15).Arc(0.1, 90)
    wp.Line(0.3).Arc(0.1, 90)
    wp.Arc(0.1, -180).Line(0.3).Arc(0.1,-180)
    wp.Arc(0.1, 90).Line(0.3).Arc(0.1, 90)
    wp.Line(0.15)
    wp.Rotate(-90).Line(0.1).Rotate(-90)
    wp.Line(0.6)
    wp.Reverse()
    synapse = wp.Face()

    # synapse.edges.col = (0,0,0)
    # synapse.edges.name = "inner_bnd"

    # for i in range(11):
    #     synapse.edges[i+1].name = "membrane"
    # synapse.faces.name = "inner_space"
    # synapse.vertices[1].name = 'membrane_bnd'
    # synapse.vertices[24].name = 'membrane_bnd'
    # synapse.vertices[2].name = 'membrane_bnd'
    # synapse.vertices[23].name = 'membrane_bnd'

    for i in range(7):
        synapse.edges[i+3].name = "membrane"
    synapse.faces.name = "inner_space"
    synapse.vertices[5].name = 'membrane_bnd'
    synapse.vertices[20].name = 'membrane_bnd'
    synapse.vertices[6].name = 'membrane_bnd'
    synapse.vertices[19].name = 'membrane_bnd'

    if external:
        # Using a 1x1 square as background and subtract the synapse profile to
        # create the external space and naming the components

        wp2 = occ.WorkPlane()
        wp2.Rectangle(1, 1)
        face = wp2.Face()
        face.edges.Min(occ.X).name = "outer_bnd"
        face.edges.Max(occ.X).name = "outer_bnd"
        face.edges.Max(occ.Y).name = "outer_bnd"

        # subtraction of the background to create the outer space
        ambient = face - synapse
        ambient.faces.name = "outer_space"

        # Merging the two spaces mantaining the interface between the two
        total = occ.Glue([synapse, ambient])

    else:

        total = synapse

    geo = occ.OCCGeometry(total, dim = 2)
    mesh = Mesh(geo.GenerateMesh(maxh=maxh))
    mesh.Curve(order_g)
    return mesh, geo

mesh, _ = generate_synapse2d(maxh=0.02)

print(mesh.GetMaterials())
print(mesh.GetBoundaries())
print(mesh.GetBBoundaries())

'''
Definition of the time parameters
'''
T = 0.016
dt = Parameter(0.0001)
t = Parameter(0)

'''
Definition of the simulation
'''
from cosmos.solvers.simulation import MovingSimulation
simulation = MovingSimulation(mesh=mesh, dt=dt, t=t, T=T, verbose = 2)

# Adding Willmore flow to the internal square
from cosmos.solvers.willmore import StabWillmoreSolver
displ_solver = StabWillmoreSolver(domain = 'membrane', clamped_bnd = 'membrane_bnd')
displ_solver.SaveSolution(folderpath = './coupled_results/', filename = 'protospine2d', n_steps = 10)

# Adding the Willmore flow motion to the overall simulation
simulation.AddMotionSolver(displ_solver)
def displ():
    return displ_solver.get_solution()[0]
simulation.AddMotion(function=displ, domain = 'membrane')

# Adding ADRsolvers in the different domains
from cosmos.solvers.adr import ADRSolver
from cosmos.utils.manufactured_solution_tools import gradient

adr_solver = ADRSolver(linear = True)
simulation.AddSolver(adr_solver)


# Solver 1
u_ex1 = cos(pi*x)*sin(pi*y)*cos(t)
d1 = 1 + x**2
c1 = 1 + x**2
b1 = CF((sin(x), 1))
flux1 = b1*u_ex1 - d1*gradient(u_ex1, Id(2))
rhs1 = (u_ex1.Diff(t) + Trace(gradient(flux1, Id(2))) + c1*u_ex1).Compile()
neu_b1 = {'.*': b1*u_ex1}
neu_d1 = {'.*': - d1*gradient(u_ex1, Id(2))}
adr_solver.AddSpecie(VorB = VOL,
                rhs = rhs1,
                advection = b1,
                diffusion = d1,
                reaction = c1,
                u0 = u_ex1,
                neu_d = neu_d1,
                neu_b = neu_b1,
                domain = 'inner_space')

# # Solver 1
# u_ex2 = cos(pi*x)*cos(t)
# d2 = 1 + x**2
# c2 = 1 + t**2
# b2 = CF((sin(x), 2))
# flux2 = b2*u_ex2 - d2*gradient(u_ex2, Id(2))
# rhs2 = (u_ex2.Diff(t) + Trace(gradient(flux2, Id(2))) + c2*u_ex2).Compile()
# dir_b2 = {'interface': u_ex2}
# dir_d2 = {'interface': u_ex2}
# neu_b2 = {'r|l|t|b': b2*u_ex2}
# neu_d2 = {'r|l|t|b': - d2*gradient(u_ex2, Id(2))}
# adr_solver.AddSpecie(VorB = VOL,
#                 rhs = rhs2,
#                 advection = b2,
#                 diffusion = d2,
#                 reaction = c2,
#                 u0 = u_ex2,
#                 neu_d = neu_d2,
#                 neu_b = neu_b2,
#                 dir_d = dir_d2,
#                 dir_b = dir_b2,
#                 domain = 'outer')

adr_solver.SaveSolution(folderpath = './coupled_results/', filename = 'spine2d_adr', n_steps = 10)

simulation.run()

adr_solver.draw_solution()
displ_solver.draw_solution()

('inner_space',)
('default', 'default', 'default', 'membrane', 'membrane', 'membrane', 'membrane', 'membrane', 'membrane', 'membrane', 'default', 'default', 'default', 'default')
('default', 'default', 'default', 'membrane_bnd', 'default', 'default', 'default', 'default', 'default', 'default', 'membrane_bnd', 'default', 'default', 'default')
------------------------------------------------------------
The mesh is a  2D mesh
The number of:
     vertices is:  623
     edges is:  1701
     faces is:  1079
     facets is:  1701
     elements is:  1079 ( = 0 if hypersurface)
Regions of co-dimension  0 :
  - There is(are)  1 region(s) called  inner_space
Regions of co-dimension  1 :
  - There is(are)  7 region(s) called  default
  - There is(are)  7 region(s) called  membrane
Regions of co-dimension  2 :
  - There is(are)  12 region(s) called  default
  - There is(are)  2 region(s) called  membrane_bnd
------------------------------------------------------------ 

---------------------------

Running Simulation...: 100%|█████████████| 160/160 [00:06<00:00, 25.49it/s]


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…